In [ ]:
#!pip install gstools
#!pip install georasters
# !pip install geemap
!pip install pingouin
!pip install contextily
#!pip install geopandas
#!pip install translate
!pip install -U deep-translator
from deep_translator import DeeplTranslator#ChatGptTranslator,
import os
from dotenv import load_dotenv
load_dotenv()
#from translate import Translator
#from googletrans import Translator
# Initialize the translator
#translator= Translator(provider='microsoft',to_lang='en')

from google.colab import drive
import io
import sys
from contextlib import redirect_stdout
import contextily as cx
import os
os.environ['USE_PYGEOS'] = '0'
with io.StringIO() as buf, redirect_stdout(buf):
   drive.mount('/content/drive')
   output = buf.getvalue()

In [ ]:
!pip install cartopy

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import numpy as np
#import gstools as gs
import matplotlib.pyplot as plt
# import ee
# import geemap
import requests
from shapely.geometry import Polygon
from shapely.geometry import Point
#import tensorflow as tf
#from tensorflow import keras
#from tensorflow.keras import layers
from datetime import datetime
from dateutil.relativedelta import relativedelta
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import pingouin as pg
#from google.colab import drive
#drive.mount('/content/drive')
#ee.Authenticate()
#ee.Initialize(project='ee-jolejua')

# Alistamiento de datos para correlaciones

Se corre cuando se tiene una nueva base de datos para alistar el archivo, sólo se debe hacer una vez, y dicho archivo queda guardado, mostrandose en el siguiente bloque

In [ ]:
dfg=gpd.read_file(r'/content/drive/Shareddrives/PIGCC/Satelite/Scripts/Cobertura_de_la_tierra_100K_Periodo_2020_limite_administrativo/shape Limite admin/e_cobertura_tierra_2020_admin.shp')#8\COBERTURAS CORINE 2018\shape coberturas 2018\cobertura_tierra_clc_2018.shp')
dfg.plot('nivel_1',legend=True)

In [ ]:
dfg.columns

In [ ]:
mydict1={'1':'1, Territorios artificializados','2':'2. Territorios agrícolas','3':'3. Bosques y áreas seminaturales','4':'4. Áreas húmedas','5':'5. Superficies de agua'}
for i in mydict1.keys():
  mydict1[i]=DeeplTranslator(api_key=os.getenv("DEEPL_API_KEY"), source="es", target="en", use_free_api=True).translate(mydict1[i])
mydict1

In [ ]:
np.array(list(mydict1.values()))

In [ ]:
import csv

with open('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/Nivel3.csv', mode='r') as infile:
    reader = csv.reader(infile)
    with open('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/N3.csv', mode='w') as outfile:
        writer = csv.writer(outfile)
        mydict = {rows[1]:DeeplTranslator(api_key=os.getenv("DEEPL_API_KEY"), source="es", target="en", use_free_api=True).translate(rows[0]) for rows in reader}
#mydict = np.array([translator.translate(text, dest='es').text for text in mydict])
mydict

In [ ]:
dfg['Leyenda_1']=dfg['nivel_1'].map(mydict1)
dfg[['Leyenda_1','nivel_1']]

In [ ]:
dfg['Leyenda_3']=dfg['nivel_3'].map(mydict)
dfg[['Leyenda_3','nivel_3']]

In [ ]:
encoded_df = pd.get_dummies(dfg, columns=["Leyenda_3"])  # Encode the "leyenda 3" column
encoded_df1 = pd.get_dummies(dfg, columns=["Leyenda_1"])  # Encode the "Leyenda 1" column
#print(encoded_df1.columns)
encoded_df=pd.concat([encoded_df,encoded_df1[encoded_df1.columns[-5:]]], axis=1)
encoded_df['groupb']=0
encoded_df

In [ ]:
encoded_df1.columns[-5:]

In [ ]:
df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/colombia_prom_2020_filtered.csv')
sns.scatterplot(data=df_s,x='longitude',y='latitude',s=0.5)#,hue='CH4_column_volume_mixing_ratio_dry_air_bias_corrected')

In [ ]:
#df_s.groupby(['latitude','longitude']).mean('CH4_column_volume_mixing_ratio_dry_air_bias_corrected').to_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom.csv')

In [ ]:
df_s

In [ ]:
df_s['coordinate_x']=df_s['longitude'].apply(lambda x: [x])
df_s['coordinate_y']=df_s['latitude'].apply(lambda x: [x])
df_s['coordinates']=df_s['coordinate_x']+df_s['coordinate_y']
df_s

In [ ]:
delta_y=0.01
delta_x=0.01
df_s['geometry'] = df_s['coordinates'].apply(
    lambda x: Polygon([
        (x[0] - delta_x, x[1] - delta_y),
        (x[0] - delta_x, x[1] + delta_y),
        (x[0] + delta_x, x[1] + delta_y),
        (x[0] + delta_x, x[1] - delta_y)
    ]))# Corregir el delta

In [ ]:
l_in=list(encoded_df.columns)
while not l_in[0]=='Leyenda_3_1.1.1. Tejido urbano continuo':
    l_in.pop(0)#(list(dfg.columns))
l_in.remove('groupb')
for i in l_in:
    encoded_df[i]=encoded_df[i]*encoded_df['SHAPE_Area']
df_s.loc[:,l_in]=None
df_s['Leyenda_1_4. Áreas húmedas'][0]

In [ ]:
for j in df_s.index:
    #print(j)
    f=encoded_df[df_s['geometry'][j].intersects(encoded_df['geometry'])]
    f1=f.groupby('groupb')[l_in].aggregate('sum') # Se suma el área total intersectada
    if not len(f1)==0:
        #print(j)
        for i in l_in:
            df_s.loc[j, i]=f1[i].iloc[0].copy()

In [ ]:
df_s

In [ ]:
#Eliminar zonas de agua para evitar glint

In [ ]:
df_s.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/corr_land_2020.csv')

#df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/glint_intersect.csv')

# Análisis de correlaciones

In [ ]:
df_corr=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/corr_land_2020_old.csv',index_col=0)
df_corr[(~(df_corr['Leyenda_1_5. Superficies de agua']>0))*(~(df_corr['Leyenda_1_4. Áreas húmedas']>0))]#+

In [ ]:
df_corr=df_corr[(~(df_corr['Leyenda_1_5. Superficies de agua']>0))*(~(df_corr['Leyenda_1_4. Áreas húmedas']>0))]
sns.scatterplot(data=df_corr,x='longitude',y='latitude',s=1)
df_corr

In [ ]:
l_in=list(df_corr.columns)
#l_in.remove(['longitude', 'latitude','CH4_column_volume_mixing_ratio_dry_air_bias_corrected', 'coordinate_x','coordinate_y', 'coordinates', 'geometry'])
while not l_in[0]=='Leyenda_3_1.1.1. Tejido urbano continuo':
    l_in.pop(0)#(list(dfg.columns))
l_in

In [ ]:
correlaciones=df_corr.corr(numeric_only=True)['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
correlaciones

In [ ]:
print(len(correlaciones))
correlaciones.sort_values(ascending=False).head(10)


In [ ]:
d={}
for i in l_in:
    #df_filtered = df_corr[df_corr[i] >1e-15]
    df_corr.loc[df_corr[i] ==0, i] = np.nan
    #transformed, lambda_ = stats.boxcox(df_corr[i])
    #cor=df_filtered['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].corr(df_filtered[i])
    #d[i]=cor
    sns.scatterplot(x=i,y='CH4_column_volume_mixing_ratio_dry_air_bias_corrected',data=df_corr)
    plt.show()

In [ ]:
corr_final=pg.pairwise_corr(df_corr,columns=[['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'],l_in])
corr_final.sort_values(by=['p-unc'])[['X', 'Y', 'n', 'r', 'p-unc','BF10']].head(60)

In [ ]:
df_sorted = corr_final.reindex(corr_final['r'].abs().sort_values(ascending=False).index)
df_sorted#[(df_sorted['p-unc']<0.01)*(df_sorted['n']>20)].head(60)

In [ ]:
#dfg.groupby('nivel_3').nunique()['leyenda']

In [ ]:
df_corr.loc[df_corr['Leyenda_1_2. Territorios agrícolas'] !=0 ]

In [ ]:
df_sorted['abs']=df_sorted['r'].abs()
df_sorted.to_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/correlaciones_fil.csv')

In [ ]:
import statsmodels
#df_cor_filtered['p-val'].to_numpy()
p_corrected=statsmodels.stats.multitest.multipletests(df_sorted['p-unc'],method='fdr_bh')
#p_corrected=statsmodels.stats.multitest.fdrcorrection(p_vals)
df_sorted[p_corrected[0]*(df_sorted['n']>20)]

In [ ]:
df_cor_filtered=df_sorted.copy()
df_cor_filtered['reject_bh']=p_corrected[0]
df_cor_filtered['reject_cb']=df_cor_filtered['p-unc']<p_corrected[3]
df_cor_filtered['p_corregido']=p_corrected[1]
df_cor_filtered

# Clusters

Se corre algoritmo K-means para encontrar las regiones del país y poder hacer las respectivas divisiones para estudio

In [ ]:
# !pip install torch
# !pip install sklearn

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.svm import LinearSVC
from sklearn import linear_model
from sklearn import model_selection as ms

In [ ]:
r = pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_filtered.csv', usecols=(2,3,5),delimiter=',') # se carga el archivo .csv
r

In [ ]:
sns.scatterplot(data=r,x='longitude',y='latitude',s=0.5)

In [ ]:
X= np.loadtxt('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_filtered.csv',skiprows=1,delimiter=',', usecols=(2,3,5)) # se carga el archivo .csv
X -= np.mean(X,axis=0) # Se deben normalizar las variables para que sean comparables# Se deben normalizar las variables para que sean comparables
X /= np.std(X,axis=0)

X.shape

In [ ]:
alpha_star=3
kmeans = KMeans(n_clusters=alpha_star, random_state=0, n_init='auto').fit(X)
labe=kmeans.labels_
r['label']=labe
r
sns.scatterplot(data=r,x='longitude',y='latitude',hue='label',s=1,palette=sns.color_palette("tab10")) #Imagen para artículo

In [ ]:
sns.scatterplot(data=r,x='longitude',y='latitude',hue='label',s=1,palette=sns.color_palette("tab10")) #Imagén para artículo

# Clasificación

Debido a que anteriormente, ningún clasificador hizo un buen trabajo, y simplemente tomaba toda la región del país como perteneciente a una misma intancia, se busca crear una red neuronal que haga la tarea de clasificación.

In [ ]:
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
y=labe
X_train, X_test, Y_train, Y_test = train_test_split(X[:,[0,1]], y, test_size=0.33)

In [ ]:
X_train.shape

In [ ]:
class Data(Dataset):
  def __init__(self, X_train, y_train):
    # need to convert float64 to float32 else
    # will get the following error
    # RuntimeError: expected scalar type Double but found Float
    self.X = torch.from_numpy(X_train.astype(np.float32))
    # need to convert float64 to Long else
    # will get the following error
    # RuntimeError: expected scalar type Long but found Float
    self.y = torch.from_numpy(y_train).type(torch.LongTensor)
    self.len = self.X.shape[0]

  def __getitem__(self, index):
    return self.X[index], self.y[index]
  def __len__(self):
    return self.len

In [ ]:
traindata = Data(X_train, Y_train)
print(traindata[25])

In [ ]:

# number of features (len of X cols)
input_dim = 2
# number of hidden layers
hidden_layers = 25
#hidden_layers2 = 125
# number of classes (unique of y)
output_dim = 3
class Network(nn.Module):
    def __init__(self,hidden_layers):
        super(Network, self).__init__()
        self.linear1 = nn.Linear(input_dim, hidden_layers)
        #self.linearh1=nn.Linear(hidden_layers,hidden_layers2)
        self.linear2 = nn.Linear(hidden_layers, output_dim)
    def forward(self, x):
        x = F.leaky_relu(self.linear1(x))
        #x = torch.sigmoid(self.linearh1(x))
        x = self.linear2(x)
        #x = F.softmax(x)
        return x


In [ ]:
epochs = 40
def run_epoch(epoch,ta,clf):
    criterion = nn.MultiMarginLoss()#nn.MultiMarginLoss()#nn.HingeEmbeddingLoss#nn.MultiLabelMarginLoss()##
    optimizer = torch.optim.Adam(clf.parameters(), lr=ta)
    for epoch in range(epochs):
      running_loss = 0.0
      for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        # set optimizer to zero grad to remove previous epoch gradients
        optimizer.zero_grad()
        # forward propagation
        outputs = clf(inputs)
        loss = criterion(outputs, labels)
        # backward propagation
        loss.backward()
        # optimize
        optimizer.step()
        running_loss += loss.item()
      # display statistics
    print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.5f}')

In [ ]:
def test_val(clf):
    correct, total = 0, 0
    # no need to calculate gradients during inference
    with torch.no_grad():
      for data in testloader:
        inputs, labels = data
        # calculate output by running through the network
        outputs = clf(inputs)
        # get the predictions
        __, predicted = torch.max(outputs.data, 1)
        # update results
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Accuracy of the network on the {len(testdata)} test data: {100 * correct // total} %')
    return correct/total

In [ ]:
def parameters_train(ep,ta,hl):
    #nn_arch(hidden_layers)
    clf = Network(hl)
    epochs = ep
    lr=ta
    run_epoch(epoch,lr,clf)
    score=test_val(clf)
    return score

In [ ]:
#Se busca el mejor parámetro para la cantidad de capas ocultas
hd=np.arange(2,30,5)
epoch=40
lr=0.005
val_scores_hl=[]
batch_size = 20000
cv=3
for hidden_layers in hd:
    print(hidden_layers)
    val_scores_hl_cv=[]
    for j in range(cv):
        X_train, X_test, Y_train, Y_test = train_test_split(X[:,[0,1]], y, test_size=0.33)
        traindata = Data(X_train, Y_train)
        trainloader = DataLoader(traindata, batch_size=batch_size,
                         shuffle=True, num_workers=2)
        testdata = Data(X_test, Y_test)
        testloader = DataLoader(testdata, batch_size=batch_size,
                                shuffle=True, num_workers=2)
        val_scores_hl_cv.append(parameters_train(epoch,lr,hidden_layers))
    val_scores_hl.append(np.mean(val_scores_hl_cv))

In [ ]:
#Se encuentra la mejor cantidad de puntos en la capa oculta
ind = np.argmax(val_scores_hl)
hl_star = 10#hd[ind] #Resultó ser 13
print('HL_star =', hl_star)

plt.plot(hd,val_scores_hl)
plt.plot(np.ones(11)*hl_star,np.arange(0,1.1,0.1),'--r')
#plt.xlim(0.9,1)
plt.ylim(0.9,1)
plt.xlabel('HL')
plt.ylabel('Mean Cross-Validation Accuracy')
plt.grid(True)
plt.show()

In [ ]:
# Se busca el mejor valor para la tasa de aprendizaje
epoch=40
lr=np.arange(1e-5,0.01,5e-4)
val_scores_lr=[]
batch_size = 500
cv=4
for i in lr:
    #print(clf.parameters)
    print(i)
    val_scores_lr_cv=[]
    for j in range(cv):
        X_train, X_test, Y_train, Y_test = train_test_split(X[:,[0,1]], y, test_size=0.33)
        traindata = Data(X_train, Y_train)
        trainloader = DataLoader(traindata, batch_size=batch_size,
                         shuffle=True, num_workers=2)
        testdata = Data(X_test, Y_test)
        testloader = DataLoader(testdata, batch_size=batch_size,
                                shuffle=True, num_workers=2)
        val_scores_lr_cv.append(parameters_train(epoch,i,hl_star))
    val_scores_lr.append(np.mean(val_scores_lr_cv))

In [ ]:
ind = np.argmax(val_scores_lr)
lr_star = lr[ind]
print('LR_star =', lr_star)

plt.plot(lr,val_scores_lr)
plt.plot(np.ones(11)*lr_star,np.arange(0,1.1,0.1),'--r')
#plt.xlim(0.95,1)
plt.ylim(0.95,1)
plt.xlabel('Learning Rate')
plt.ylabel('Mean Cross-Validation Accuracy')
plt.grid(True)
plt.show()

In [ ]:
#plt.scatter(inputs[:,0],inputs[:,1],c=predicted)
parameters_train(40,0.005,15)

In [ ]:
# save the trained model
hl_star=15
clf = Network(hl_star)
criterion = nn.MultiMarginLoss()#nn.MultiMarginLoss()#nn.HingeEmbeddingLoss#nn.MultiLabelMarginLoss()##
optimizer = torch.optim.Adam(clf.parameters(), lr=lr_star)
epochs = 40
for epoch in range(epochs):
  running_loss = 0.0
  for i, data in enumerate(trainloader, 0):
    inputs, labels = data
    # set optimizer to zero grad to remove previous epoch gradients
    optimizer.zero_grad()
    # forward propagation
    outputs = clf(inputs)
    loss = criterion(outputs, labels)
    # backward propagation
    loss.backward()
    # optimize
    optimizer.step()
    running_loss += loss.item()
  # display statistics
  print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.5f}')
PATH = '/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/mymodel_prueba2.pth'
torch.save(clf.state_dict(), PATH)

In [ ]:
# load the trained model
clf = Network(hl_star)
clf.load_state_dict(torch.load(PATH))

In [ ]:
# mapa con la clasificación de resultado
#sns.scatterplot(x=inputs[:,0],y=inputs[:,1])#],hue='label',s=1,palette=sns.color_palette("tab10"))
batch_size=len(y)
wdata = Data(X[:,[0,1]], y)
dataloader = DataLoader(wdata, batch_size=batch_size,
                                shuffle=True, num_workers=2)
y_pred=[]
for data in dataloader:
    inputs, labels = data
    # calculate output by running through the network
    outputs = clf(inputs)
    # get the predictions
    __, predicted = torch.max(outputs.data, 1)
    y_pred.append(predicted)
#r['region']=predicted
# fig, ax = plt.subplots()
# for label in [0, 1,2,3]:
#     mask = (y_pred == label)
#     ax.scatter(inputs[mask, 0], inputs[mask, 1],s=0.3)
#plt.scatter(inputs[:,0],inputs[:,1],c=predicted)


In [ ]:
plt.scatter(inputs[:,1],inputs[:,0],c=predicted)

In [ ]:
r['region']=predicted
sns.scatterplot(data=r,x='longitude',y='latitude',hue='region')

# Series de tiempo

In [ ]:
#Crear los dataframes a partir de la clasificación
df_car.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Caribe')
df_valle.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Valle')
df_amazon.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Amazonía')
df_orin.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Orinoquía')
df_pacif.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Pacífico')
#df_bog.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Bogotá')
df.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].plot(label='Colomibia')
plt.legend()
plt.ylabel('Methane concentration [ppb]')
plt.xlabel('Date')
plt.title('Methane Time Series for Colombian regions')
plt.grid()

In [ ]:
series1=df.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
series1_std=df.groupby(['year','month']).std()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
series2=df_amazon.groupby(['year','month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
series2_std=df_amazon.groupby(['year','month']).std()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
df_mean=pd.merge(series1,series2, on=['year','month'])
df_std=pd.merge(series1_std,series2_std, on=['year','month'])
df_mean

In [ ]:
import scipy.stats as stats

series1=df_mean['CH4_column_volume_mixing_ratio_dry_air_bias_corrected_x']
series1_std=df_std['CH4_column_volume_mixing_ratio_dry_air_bias_corrected_x']
series2=df_mean['CH4_column_volume_mixing_ratio_dry_air_bias_corrected_y']
series2_std=df_std['CH4_column_volume_mixing_ratio_dry_air_bias_corrected_y']
t_stat, p_value = stats.ttest_ind_from_stats(
    mean1=np.array(series1),
    std1=np.array(series1_std),
    nobs1=len(series1),
    mean2=np.array(series2),
    std2=np.array(series2_std),
    nobs2=len(series2),
    alternative='greater'
)
print(sum(p_value*100>5)/len(series1)*100) #porcentaje en el que no hay evidencia suficiente para rechazar la hipótesis
print(sum(p_value*100<0.1)/len(series1)*100) # Porcentaje en el que hay evidencia indistutible para rechazar la hipótesis nula
#print(max(p_value)*100)
i_p=np.argwhere(p_value>0.05)
ind[i_p]

In [ ]:
t_stat, p_value = stats.ttest_ind(series1, series2, equal_var=False,alternative='greater')
print(p_value*100) # comparación de medias, Evidencia indiscutible

In [ ]:
#Se sospecha que no siempre es menor el metano en el Amazonas, por lo que se hace prueba de hipótesis entre promedios multianuales
series1m=df.groupby(['month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
series1_stdm=df.groupby(['month']).std()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
series2m=df_amazon.groupby(['month']).mean()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
series2_stdm=df_amazon.groupby(['month']).std()['CH4_column_volume_mixing_ratio_dry_air_bias_corrected']
t_statm, p_valuem = stats.ttest_ind_from_stats(
    mean1=np.array(series1m),
    std1=np.array(series1_stdm),
    nobs1=len(series1m),
    mean2=np.array(series2m),
    std2=np.array(series2_stdm),
    nobs2=len(series2m),
    alternative='greater'
)
print(p_valuem*100<5)
print(p_valuem*100) # No se puede refutar la hipótesis de que el Amazonas tenga menores niveles de metano en los meses de Enero, Febrero, Marzo y Diciembre

In [ ]:
df_tsc=pd.read_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/CH4_completo.csv',skiprows=1)
df_tsc=df_tsc[(df_tsc['value']>1850)*(df_tsc['year']>2015)]
df_tsc.groupby(['year','month'])['value'].mean().plot()
plt.ylabel('Methane concentration [ppb]')
plt.xlabel('Month')
plt.title('Methane Time Series reference value')
plt.grid()


In [ ]:
df_g=pd.read_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/ch4_mm_gl.csv',skiprows=45)
df_g=df_g[df_g['year']>2015]
df_g.groupby(['year','month'])['average'].mean().plot()
plt.ylabel('Methane concentration [ppb]')
plt.xlabel('Month')
plt.title('Methane Time Series global value')
plt.grid()

In [ ]:
df_c_b=pd.merge(df,df_c, on=['year','month'])
df_c_r=pd.merge(df_c_b,df_g,on=['year','month'])
df_c_r

In [ ]:
df_c_r.groupby(['year','month'])['CH4_column_volume_mixing_ratio_dry_air_bias_corrected'].mean().plot(label='Colombia')
df_c_r.groupby(['year','month'])['average'].mean().plot(label='Global')
df_c_r.groupby(['year','month'])['value'].mean().plot(label='Barbados')
plt.legend()
plt.ylabel('Methane concentration [ppb]')
plt.xlabel('Date')
plt.title('Methane Time Series for Colombia vs reference value')
plt.grid()

# Detección de anomalías

In [ ]:
X= np.loadtxt(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020_filtered.csv',skiprows=1, usecols=(1,2,4),delimiter=',')
X

In [ ]:
plt.scatter(X[:,0],X[:,1],s=0.0001)

In [ ]:
def prepData(processed_data,iSplit=0.5):
    # gaia_vars=['phot_g_mean_mag','phot_bp_mean_mag','phot_rp_mean_mag']
    # var0=(igaia_data[gaia_vars[0]]-idist.distmod).value
    # var1=(igaia_data[gaia_vars[1]]-idist.distmod).value
    # var2=(igaia_data[gaia_vars[2]]-idist.distmod).value
    # var3=igalcen.x.value
    # var4=igalcen.y.value
    # var5=igalcen.z.value
    # var6=igalcen.v_x.value
    # var7=igalcen.v_y.value
    # var8=igalcen.v_z.value

    #processed_data = np.vstack((var0,var1,var2,var3,var4,var5,var6,var7,var8))
    # processed_data = np.vstack((var0,var1,var2,var3,var4,var5,var6,var7,var8))
    # processed_data = processed_data.T
    # processed_data = processed_data[~np.isnan(processed_data).any(axis=1)]
    processed_data_raw = processed_data.copy()

    #normalize the data
    processed_data -= np.mean(processed_data,axis=0)
    processed_data /= np.std(processed_data,axis=0)
    processed_data = processed_data[~np.isnan(processed_data).any(axis=1)]

    #pytorch the layer
    tprocessed_data = torch.tensor(processed_data).float()
    processed_data_raw = processed_data_raw[~torch.any(tprocessed_data.isnan(),dim=1)]
    #galcen_clean       = igalcen[~torch.any(tprocessed_data.isnan(),dim=1)]
    n_samples=processed_data.shape[0]
    indices = np.arange(n_samples)
    #split
    X_train, X_test, Y_train, Y_test = train_test_split(processed_data, indices, test_size=iSplit)
    maxindex       = int(len(processed_data)*iSplit)
    trainset       = torch.tensor(X_train).float()
    trainset       = trainset[~torch.any(trainset.isnan(),dim=1)]
    testset        = torch.tensor(X_test).float()
    testset        = testset[~torch.any(testset.isnan(),dim=1)]
    print(processed_data_raw.shape,testset.shape,trainset.shape)
    return testset,trainset,processed_data_raw,tprocessed_data, Y_train, Y_test #,galcen_clean

#btestset,btrainset,bprocessed_data_raw,btprocessed_data,bgalcen_clean=prepData(gaia_data,dist,galcen)
btestset,btrainset,bprocessed_data_raw,btprocessed_data,index_train,index_test=prepData(X)

In [ ]:
#autoencoder para que la red neuronal aprenda las características más relevantes de los datos
class MLP(nn.Module):
    def __init__(self,n_inputs,n_outputs):
        super(MLP, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_inputs, 20),
            nn.LeakyReLU(),
            nn.Linear(20, 6),
            nn.ReLU(),
            nn.Linear(6, 2),
            nn.ReLU(),
            nn.Linear(2, 6),
            nn.ReLU(),
            nn.Linear(6, 20),
            nn.ReLU(),
            nn.Linear(20, n_outputs),
        )

    def forward(self, x):
        x = self.layers(x)
        return x

def train(x,y,net,loss_func,opt,sched,nepochs):
    net.train(True)
    for epoch in range(nepochs):
        prediction = net(x)
        opt.zero_grad()
        loss = loss_func(prediction,y)
        loss.backward()
        opt.step()
        if epoch % 500 == 0:
            print('[%d] loss: %.4f ' % (epoch + 1, loss.item()  ))
    #sched.step()
    return

basicmodel     = MLP(btrainset.shape[1],1)#btrainset.shape[1])
optimizer = torch.optim.Adam(basicmodel.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1, last_epoch=-1, verbose=False)
loss_fn   =  nn.MSELoss(reduction='sum')

In [ ]:
train(btrainset,btrainset[:,2:],basicmodel,loss_fn,optimizer,scheduler,5001)

In [ ]:
btrainset.shape

In [ ]:
basicmodel.train(False)
boutput=basicmodel(btestset)
btestloss=torch.sum((btestset-boutput)**2,axis=1)
#btestloss2=torch.sum((btestset-boutput)**1,axis=1)
plt.hist(btestloss[btestloss < 100].detach().numpy(),density=True,bins=30)
plt.yscale('log')
plt.xlabel('loss')
plt.ylabel('pdf')
plt.show()

varlabels=['longitude','latitude','CH4']#['Mag','B-Mag','R-Mag','x','y','z','vx','vy','vz']
_,bins,_=plt.hist(btestset[:,2].detach().numpy(),density=True,alpha=0.5,label='Input')
plt.hist(boutput.detach().numpy(),density=True,alpha=0.5,bins=bins,label='Output')
plt.xlabel(varlabels[2])
plt.legend()
# fig, ax = plt.subplots(3, 3, figsize=(20, 20))
# for var in range(btestset.shape[1]):
#     _,bins,_=ax[var//3,var % 3].hist(btestset[:,var].detach().numpy(),density=True,alpha=0.5,label='Input')
#     ax[var//3,var % 3].hist(boutput [:,var].detach().numpy(),density=True,alpha=0.5,bins=bins,label='Output')
#     ax[var//3,var % 3].set_xlabel(varlabels[var])
#     ax[var//3,var % 3].legend()

In [ ]:
def plotAnomaly(iCut, iRaw, iLoss, i_test, iLoss2, maxlosscolor=20, figsize=(20, 8),
                save_plot=False, filename='anomaly_plot.png'):
    """
    Plot anomalies in methane data with enhanced visualization and map background.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import cartopy.crs as ccrs
    import contextily as cx
    from matplotlib.gridspec import GridSpec
    from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

    # Prepare the data
    loss = np.minimum(iLoss2, maxlosscolor)
    anomalies = (iLoss > iCut)
    btestdata_raw = iRaw[i_test]

    # Separate positive and negative anomalies
    pos_anomaly_raw = btestdata_raw[anomalies*(iLoss2>0)]
    neg_anomaly_raw = btestdata_raw[anomalies*(iLoss2<0)]
    anomaly_raw = btestdata_raw[anomalies]

    # Normalize loss values
    loss = loss/np.max(loss)
    if maxlosscolor != 20:
        loss = np.ones(loss.shape)

    # Create figure with GridSpec
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 3, figure=fig)

    # Map plot with anomalies
    ax1 = fig.add_subplot(gs[0], projection=ccrs.epsg(3857))

    # Set map extent for Colombia
    extent = [-81, -66.1, -4.5, 12.9]
    ax1.set_extent(extent, crs=ccrs.PlateCarree())

    # Plot points on map
    scat = ax1.scatter(btestdata_raw[:,0], btestdata_raw[:,1],
                      marker='.', c=loss, cmap="Spectral",
                      transform=ccrs.PlateCarree(),
                      s=5, alpha=0.6)

    # Plot anomalies
    ax1.scatter(pos_anomaly_raw[:,0], pos_anomaly_raw[:,1],
                c='#006400', s=10, alpha=0.7,
                label='Positive Anomalies',
                transform=ccrs.PlateCarree())
    ax1.scatter(neg_anomaly_raw[:,0], neg_anomaly_raw[:,1],
                c='red', s=10, alpha=0.7,
                label='Negative Anomalies',
                transform=ccrs.PlateCarree())

    # Add basemap
    cx.add_basemap(ax1, source=cx.providers.OpenStreetMap.Mapnik)

    # Add gridlines
    gl = ax1.gridlines(crs=ccrs.PlateCarree(),
                      draw_labels=True,
                      linewidth=1,
                      color='gray',
                      alpha=0.5,
                      linestyle='--')

    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 10, 'color': 'gray'}
    gl.ylabel_style = {'size': 10, 'color': 'gray'}

    # # Add legend
    # legend = ax1.legend(title='Anomalies',
    #                    loc='upper left',
    #                    bbox_to_anchor=(1.05, 1),
    # #                    markerscale=3)
    # plt.setp(legend.get_title(), fontsize=10)

    ax1.set_title('Spatial Distribution of Anomalies', pad=20)

    # Latitude-CH4 plot
    ax2 = fig.add_subplot(gs[1])
    scat2 = ax2.scatter(btestdata_raw[:,1], btestdata_raw[:,2],
                       c=loss, marker='.', cmap="Spectral",
                       s=5, alpha=0.6)
    ax2.scatter(pos_anomaly_raw[:,1], pos_anomaly_raw[:,2],
                c='#006400', s=10, alpha=0.7,
                label='Positive Anomalies')
    ax2.scatter(neg_anomaly_raw[:,1], neg_anomaly_raw[:,2],
                c='red', s=10, alpha=0.7,
                label='Negative Anomalies')
    ax2.set_xlabel("Latitude", fontsize=10)
    ax2.set_ylabel("CH₄ (ppb)", fontsize=10)
    ax2.set_title('Latitudinal Distribution of CH₄', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    # Longitude-CH4 plot
    ax3 = fig.add_subplot(gs[2])
    scat3 = ax3.scatter(btestdata_raw[:,0], btestdata_raw[:,2],
                       c=loss, marker='.', cmap="Spectral",
                       s=5, alpha=0.6)
    ax3.scatter(pos_anomaly_raw[:,0], pos_anomaly_raw[:,2],
                c='#006400', s=10, alpha=0.7,
                label='Positive Anomalies')
    ax3.scatter(neg_anomaly_raw[:,0], neg_anomaly_raw[:,2],
                c='red', s=10, alpha=0.7,
                label='Negative Anomalies')
    ax3.set_xlabel("Longitude", fontsize=10)
    ax3.set_ylabel("CH₄ (ppb)", fontsize=10)
    ax3.set_title('Longitudinal Distribution of CH₄', fontsize=12)
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    # Add colorbar
    cbar = plt.colorbar(scat3, ax=ax3)
    cbar.set_label('Normalized Loss', fontsize=10)

    # Adjust layout
    plt.tight_layout()

    # Save plot if requested
    if save_plot:
        plt.savefig(filename, dpi=300, bbox_inches='tight')

    plt.show()

    return pos_anomaly_raw, neg_anomaly_raw, anomaly_raw
# def plotAnomaly(iCut,iRaw,iLoss,i_test,iLoss2,maxlosscolor=20):
#     loss = np.minimum(iLoss2,maxlosscolor)
#     anomalies=(iLoss > iCut)

#     #baseidex=len(iRaw)-len(iLoss)
#     btestdata_raw = iRaw[i_test]
#     pos_anomaly_raw=btestdata_raw[anomalies*(iLoss2>0)]
#     neg_anomaly_raw=btestdata_raw[anomalies*(iLoss2<0)]
#     anomaly_raw  = btestdata_raw[anomalies]
#     # pos_anomaly= anomaly_raw[iLoss2>0]
#     # neg_anomaly= anomaly_raw[iLoss2<0]
#     #btestgalcen   = igalcen[baseidex:]#
#     #anomaly_galcen = btestgalcen[anomalies]#
#     #print(anomaly_raw)
#     vma=1
#     vmi=-1*vma
#     loss = loss/np.max(loss)
#     if maxlosscolor != 20:
#         loss=np.ones(loss.shape)
#     print(loss.shape,btestdata_raw[:,2].shape)
#     scat=plt.scatter(btestdata_raw[:,0],btestdata_raw[:,1], marker='.',c=loss,cmap="Spectral")
#     plt.plot(anomaly_raw[:,0],anomaly_raw[:,1], marker='.', linestyle='none',c='orange')
#     plt.xlabel('$Longitude$')
#     plt.ylabel('$Latitude$')
#     #plt.ylim(15,0)
#     #plt.xlim(1,-5)
#     plt.colorbar(scat)
#     plt.show()
#     plt.figure(figsize=(16,4))
#     plt.subplot(1,3,1)
#     scat=plt.scatter(btestdata_raw[:,0],btestdata_raw[:,1], marker='.',c=loss,cmap="Spectral", vmin=vmi, vmax=vma)
#     plt.plot(pos_anomaly_raw[:,0],pos_anomaly_raw[:,1], marker='.', linestyle='none',c='#006400')
#     plt.plot(neg_anomaly_raw[:,0],neg_anomaly_raw[:,1], marker='.', linestyle='none',c="#FF0000")
#     plt.xlabel('$Longitude$')
#     plt.ylabel('$Latitude$')
#     #plt.ylim(15,0)
#     #plt.xlim(1,-5)
#     #plt.colorbar(scat)
#     #plt.show()
#     plt.subplot(1,3,2)
#     scat=plt.scatter(btestdata_raw[:,1],btestdata_raw[:,2],c=loss, marker='.',cmap="Spectral",vmin=vmi, vmax=vma)
#     plt.plot(pos_anomaly_raw[:,1],pos_anomaly_raw[:,2], marker='.', linestyle='none',c='#006400')
#     plt.plot(neg_anomaly_raw[:,1],neg_anomaly_raw[:,2], marker='.', linestyle='none',c='red')
#     #plt.plot(anomaly_raw[:,1],anomaly_raw[:,2], marker='.', linestyle='none',c='orange')
#     plt.xlabel("$Latitude$")
#     plt.ylabel("CH4[ppb]")
#     #plt.colorbar(scat)
#     #plt.show()
#     plt.subplot(1,3,3)
#     scat=plt.scatter(btestdata_raw[:,0],btestdata_raw[:,2],c=loss, marker='.', cmap="Spectral", vmin=vmi, vmax=vma)
#     plt.plot(pos_anomaly_raw[:,0],pos_anomaly_raw[:,2], marker='.', linestyle='none',c='#006400')
#     plt.plot(neg_anomaly_raw[:,0],neg_anomaly_raw[:,2], marker='.', linestyle='none',c='red')
#     #plt.plot(anomaly_raw[:,0],anomaly_raw[:,2], marker='.', linestyle='none',c='orange')
#     plt.xlabel("$Longitude$")
#     plt.ylabel("CH4[ppb]")
#     plt.colorbar(scat)
#     plt.show()
#     return pos_anomaly_raw,neg_anomaly_raw,anomaly_raw
    # scat=plt.scatter(btestdata_raw[:,6],btestdata_raw[:,7],c=loss, marker='.', cmap="viridis")
    # plt.plot(anomaly_raw[:,6],anomaly_raw[:,7], marker='.', linestyle='none',c='orange')
    # plt.xlabel("vx[pc]")
    # plt.ylabel("vy[pc]")
    # plt.colorbar(scat)
    # plt.show()

    # scat=plt.scatter(btestdata_raw[:,6],btestdata_raw[:,8],c=loss, marker='.',cmap="viridis")
    # plt.plot(anomaly_raw[:,6],anomaly_raw[:,8], marker='.', linestyle='none')
    # plt.xlabel("vx[pc]")
    # plt.ylabel("vz[pc]")
    # plt.show()

    # print(len(btestdata_raw),len(iRaw),len(igalcen),len(btestgalcen))
    # H = gp.Hamiltonian(milky_way)
    # w0_anom = gd.PhaseSpacePosition(anomaly_galcen.cartesian)
    # orbits_anom = H.integrate_orbit(w0_anom, dt=1*u.Myr, t1=0*u.Myr, t2=100*u.Myr)
    # w0_all  = gd.PhaseSpacePosition(bgalcen_clean[1==1].cartesian)
    # orbits_all  = H.integrate_orbit(w0_all,  dt=1*u.Myr, t1=0*u.Myr, t2=100*u.Myr)

    # zmax_all = orbits_all.zmax(approximate=True)
    # zmax_anom = orbits_anom.zmax(approximate=True)
    # print("ZMax mean:",zmax_anom[~np.isnan(zmax_anom.value)].mean(),len(zmax_anom),"default:",zmax_all.mean())
    # bins = np.linspace(0, 10, 50)
    # plt.hist(zmax_all.value, bins=bins, alpha=0.4, density=True, label='all')
    # plt.hist(zmax_anom.value, bins=bins, alpha=0.4, density=True, label='anom')
    # plt.legend(loc='best', fontsize=14)
    # plt.yscale('log')
    # plt.xlabel(r" zmax" + " [{0:latex}]".format(zmax_all.unit))
    # plt.show()

    # zmax_all  = orbits_all.eccentricity()
    # zmax_anom = orbits_anom.eccentricity()
    # print("Ecc mean:",zmax_anom[~np.isnan(zmax_anom.value)].mean(),len(zmax_anom),"default:",zmax_all.mean())
    # bins = np.linspace(0, 3, 50)
    # plt.hist(zmax_all.value,  bins=bins, alpha=0.4, density=True, label='all')
    # plt.hist(zmax_anom.value, bins=bins, alpha=0.4, density=True, label='anom')
    # plt.legend(loc='best', fontsize=14)
    # plt.yscale('log')
    # plt.xlabel('Eccentricity')
    # plt.show()
# cutoff 20 whole average, 14 prom 2020
btestloss2=torch.sum((btestset-boutput)**1,axis=1)
positive_anomaly,negative_anomaly,anomaly=plotAnomaly(14,bprocessed_data_raw,btestloss.detach().numpy(),index_test,btestloss2.detach().numpy())#,bgalcen_clean)


In [ ]:
#Relacionar con el uso de la tierra en puntos anomalos
encoded_df45=encoded_df[~((encoded_df['Leyenda_1_4. Wet areas']>0)+(encoded_df['Leyenda_1_5. Water surfaces']>0))]
#encoded_df45=encoded_df[encoded_df['Leyenda_3_1.3.1. Zonas de extracción minera']>0]
#encoded_df45=encoded_df.copy()

In [ ]:
#encoded_df.geometry.intersects(positive_anomaly)
# This will return a boolean Series
for i in range(len(positive_anomaly)):
  is_within = encoded_df45.geometry.contains(Point(positive_anomaly[i,0], positive_anomaly[i,1]))
  encoded_df45.loc[is_within,'groupb']=1
# encoded_df2=encoded_df45.copy()
# encoded_df2['groupb']=0
for i in range(len(negative_anomaly)):
  is_within = encoded_df45.geometry.contains(Point(negative_anomaly[i,0], negative_anomaly[i,1]))
  encoded_df45.loc[is_within,'groupb']=2
#encoded_df2[encoded_df2['groupb']==1].geometry
#encoded_df45[encoded_df45['groupb']==1].geometry

In [ ]:
l_in=list(encoded_df45.columns)
pos_level3=encoded_df45[encoded_df45['groupb']==1].groupby('groupb')[l_in[25:-6]].aggregate('sum')
pos_level1=encoded_df45[encoded_df45['groupb']==1].groupby('groupb')[l_in[-6:-3]].aggregate('sum')
#matching_polygons = gpd.sjoin(gdf, points_gdf, predicate='contains')
neg_level3=encoded_df45[encoded_df45['groupb']==2].groupby('groupb')[l_in[25:-6]].aggregate('sum')
neg_level1=encoded_df45[encoded_df45['groupb']==2].groupby('groupb')[l_in[-6:-3]].aggregate('sum')

In [ ]:
# plt.figure(figsize=(15,15))
# plt.subplot(3,3,1)
# pos_rel=pos_level1>0
# pos_columns=pos_level1.columns[pos_rel.iloc[0]]
# pos_level1=pos_level1[pos_columns]
# patches, texts, autotexts = plt.pie(pos_level1.iloc[0],
#                                   labels=[''] * len(pos_level1.columns),  # Remove direct labels
#                                   autopct='%1.1f%%',
#                                   colors=['#ff7f0e', '#2ca02c'],
#                                   startangle=90)
# # plt.legend(patches, pos_level1.columns,
# #           title="Categories",
# #           loc="center left",
# #           bbox_to_anchor=(1, 0, 0.5, 1))  # Place legend to the right
# plt.title('Positive anormal points')
# plt.axis('equal')
# plt.subplot(3,1,1)
# patches, texts, autotexts = plt.pie(neg_level1.iloc[0],
#                                   labels=[''] * len(neg_level1.columns),  # Remove direct labels
#                                   autopct='%1.1f%%',
#                                   startangle=90)
# plt.legend(patches, [i[10:] for i in neg_level1.columns],
#           title="Categories",
#           loc="center left",
#           bbox_to_anchor=(0.7, 0, 0.5, 1))  # Place legend to the right
# plt.title('Negative anormal points ')
# plt.axis('equal')
# plt.show()
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# Set publication-quality style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'font.size': 15,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.figsize': (15, 8),
    'savefig.dpi': 300,
    'savefig.format': 'pdf'
})

# Create figure
fig = plt.figure(constrained_layout=True)
gs = fig.add_gridspec(2, 2, height_ratios=[2, 1])

# First pie chart (positive anomalies)
ax1 = fig.add_subplot(gs[0])
pos_rel = pos_level1 > 1
pos_columns = pos_level1.columns[pos_rel.iloc[0]]
pos_level1_filtered = pos_level1[pos_columns]

# Professional color palette
colors = ['#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377']

wedges, texts, autotexts = ax1.pie(
    pos_level1_filtered.iloc[0],
    autopct='%1.1f%%',
    colors=colors[1:len(pos_columns)+1],
    startangle=90,
    wedgeprops={'edgecolor': 'w', 'linewidth': 1}
)

# Customize text appearance
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# Add legend with clean category names
# ax1.legend(
#     wedges,
#     [col.replace('_', ' ').title() for col in pos_columns],
#     title="Positive Anomaly Categories",
#     loc="center left",
#     bbox_to_anchor=(1, 0.5)
# )
ax1.set_title('Distribution of Positive Anomalies', fontweight='bold')

# Second pie chart (negative anomalies)
ax2 = fig.add_subplot(gs[1])
wedges, texts, autotexts = ax2.pie(
    neg_level1.iloc[0],
    autopct='%1.1f%%',
    colors=colors[:len(neg_level1.columns)],
    startangle=90,
    wedgeprops={'edgecolor': 'w', 'linewidth': 1}
)

# Customize text appearance
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# Clean up category names by removing prefix
clean_labels = [col[10:].replace('_', ' ').title() if col.startswith('neg_level1')
                else col[10:].replace('_', ' ').title() for col in neg_level1.columns]

ax2.legend(
    wedges,
    clean_labels,
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)
ax2.set_title('Distribution of Negative Anomalies', fontweight='bold')

# Add overall figure title
#fig.suptitle('Anomaly Classification Analysis', fontsize=18, fontweight='bold', y=0.98)

# Save figure
plt.savefig('anomaly_classification.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
# print("Default color cycle:", default_colors)

In [ ]:
total=pos_level3.iloc[0].sum()
pos_rel=pos_level3/total*100>0
pos_columns=pos_level3.columns[pos_rel.iloc[0]]
pos_level3=pos_level3[pos_columns]
pos_level3=pos_level3[pos_rel]
pos_rel=pos_level3/total*100>0
l=pos_level3/total*100
l.iloc[0].sort_values()

In [ ]:
# pos_rel=pos_level3/total*100>3
# pos_columns=pos_level3.columns[pos_rel.iloc[0]]
# pos_level3=pos_level3[pos_columns]
# pos_level3=pos_level3[pos_rel]
# #neg_level3=neg_level3.sort_values(by=0)
# others=total-pos_level3[pos_rel].iloc[0].sum()
# pos_level3_graph=pos_level3.iloc[0].sort_values().to_numpy()
# index=np.concatenate((np.array(['Others']),np.array([i[10:] for i in pos_level3.iloc[0].sort_values().index])))
# plt.figure(figsize=(10, 8))
# patches, texts, autotexts =plt.pie(np.concatenate((np.array([others]),pos_level3_graph)),  # Get the first (and only) row
#         labels=index,#[''] * len(pos_level3.columns),
#         autopct='%1.1f%%',  # Show percentages
#         startangle=90)
# plt.title('Positive anormal points by land use on level 3  ')
# # plt.legend(patches, pos_level3.iloc[0].sort_values().index,
# #           title="Categories",
# #           loc="center left",
# #           bbox_to_anchor=(1, 0, 0.5, 1))  # Place legend to the right
# plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
# plt.show()

import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# Set publication-quality style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
    'figure.figsize': (10, 8),
    'savefig.dpi': 300,
    'savefig.format': 'pdf'
})

# Process data
pos_rel = pos_level3/total*100 > 3
pos_columns = pos_level3.columns[pos_rel.iloc[0]]
pos_level3_filtered = pos_level3[pos_columns]
pos_level3_filtered = pos_level3_filtered[pos_rel]

# Calculate "Others" category
others = total - pos_level3_filtered.iloc[0].sum()

# Sort values for better visualization
pos_level3_graph = pos_level3_filtered.iloc[0].sort_values().to_numpy()
index = np.concatenate((np.array(['Others']),
                        np.array([i[10:].replace('_', ' ').title() for i in
                                 pos_level3_filtered.iloc[0].sort_values().index])))

# Set up figure
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

# Color palette - colorblind friendly
colors = ['#DDDDDD', '#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377']

# Create pie chart
wedges, texts, autotexts = ax.pie(
    np.concatenate((np.array([others]), pos_level3_graph)),
    labels=None,  # We'll use a legend instead
    autopct='%1.1f%%',
    startangle=90,
    colors=colors[:len(index)],
    wedgeprops={'edgecolor': 'w', 'linewidth': 1},
    pctdistance=0.85
)

# Customize text appearance
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

# Add legend with clean category names
ax.legend(
    wedges,
    index,
    title="Land Use Categories",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    framealpha=0.9,
    edgecolor='black'
)

# Add title with proper formatting
ax.set_title('Distribution of Positive Anomalies by Land Use (Level 3)',
             fontweight='bold', pad=20)

# Add footnote for methodology
plt.annotate('Note: Only categories representing >3% of total area are shown individually',
             xy=(0.5, -0.05), xycoords='axes fraction',
             ha='center', va='center', fontsize=10, fontstyle='italic')

# Equal aspect ratio ensures that pie is drawn as a circle
ax.axis('equal')

# Save figure
plt.savefig('land_use_anomalies.pdf', bbox_inches='tight')
plt.show()


In [ ]:
total=neg_level3.iloc[0].sum()
neg_rel=neg_level3/total*100>0
neg_columns=neg_level3.columns[neg_rel.iloc[0]]
neg_level3=neg_level3[neg_columns]
neg_level3=neg_level3[neg_rel]
neg_rel=neg_level3/total*100>0
l=neg_level3/total*100
l.iloc[0].sort_values()

In [ ]:
# neg_rel=neg_level3/total*100>2.3
# neg_columns=neg_level3.columns[neg_rel.iloc[0]]
# neg_level3=neg_level3[neg_columns]
# neg_level3=neg_level3[neg_rel]
# #neg_level3=neg_level3.sort_values(by=0)
# others=total-neg_level3[neg_rel].iloc[0].sum()
# neg_level3_graph=neg_level3.iloc[0].sort_values().to_numpy()
# index=np.concatenate((np.array(['Others']),np.array([i[10:] for i in neg_level3.iloc[0].sort_values().index])))
# plt.figure(figsize=(10, 8))
# patches, texts, autotexts =plt.pie(np.concatenate((np.array([others]),neg_level3_graph)),  # Get the first (and only) row
#         labels=index,#[''] * len(pos_level3.columns),
#         autopct='%1.1f%%',  # Show percentages
#         startangle=90)
# plt.title('Negative anormal points by land use on level 3  ')
# # plt.legend(patches, pos_level3.iloc[0].sort_values().index,
# #           title="Categories",
# #           loc="center left",
# #           bbox_to_anchor=(1, 0, 0.5, 1))  # Place legend to the right
# plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
# plt.show()
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl

# Set publication-quality style
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
    'figure.figsize': (10, 8),
    'savefig.dpi': 300,
    'savefig.format': 'pdf'
})

# Process data
neg_rel = neg_level3/total*100 > 4
neg_columns = neg_level3.columns[neg_rel.iloc[0]]
neg_level3_filtered = neg_level3[neg_columns]
neg_level3_filtered = neg_level3_filtered[neg_rel]

# Calculate "Others" category
others = total - neg_level3_filtered.iloc[0].sum()

# Sort values for better visualization
neg_level3_graph = neg_level3_filtered.iloc[0].sort_values().to_numpy()
index = np.concatenate((np.array(['Others']),
                        np.array([i[10:].replace('_', ' ').title() for i in
                                  neg_level3_filtered.iloc[0].sort_values().index])))

# Set up figure
fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)

# Color palette - use reds for negative anomalies (diverging from blue in positive chart)
colors = ['#DDDDDD', '#8B0000', '#A52A2A', '#CD5C5C', '#E9967A', '#F08080', '#FA8072', '#FF0000']  # Added Pure Red

# Create pie chart
wedges, texts, autotexts = ax.pie(
    np.concatenate((np.array([others]), neg_level3_graph)),
    labels=None,  # We'll use a legend instead
    autopct='%1.1f%%',
    startangle=90,
    colors=colors[:len(index)],
    wedgeprops={'edgecolor': 'w', 'linewidth': 1},
    pctdistance=0.85
)

# Customize text appearance
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

# Add legend with clean category names
ax.legend(
    wedges,
    index,
    title="Land Use Categories",
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=True,
    framealpha=0.9,
    edgecolor='black'
)

# Add title with proper formatting
ax.set_title('Distribution of Negative Anomalies by Land Use (Level 3)',
             fontweight='bold', pad=20)

# Add footnote for methodology
plt.annotate('Note: Only categories representing >2.3% of total area are shown individually',
             xy=(0.5, -0.05), xycoords='axes fraction',
             ha='center', va='center', fontsize=10, fontstyle='italic')

# Equal aspect ratio ensures that pie is drawn as a circle
ax.axis('equal')

# Save figure
plt.savefig('negative_land_use_anomalies.pdf', bbox_inches='tight')
plt.show()

# Suplemental
Se debe correr la sección alistamiento de datos hastala línea donde se carga df_s y se muestra un mapa

In [ ]:
encoded_df45=encoded_df[(encoded_df['Leyenda_1_4. Áreas húmedas']>0)+(encoded_df['Leyenda_1_5. Superficies de agua']>0)]#tabla para eliminar el glint en la plataforma continental

In [ ]:
encoded_df45

In [ ]:
#df_g es el promedio de todos los año filtrado sin cuerpo de agua para evitar el glint
#df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom.csv')
df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020.csv')
#df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020
#df_g=df_s[df_s['Leyenda_1_4. Áreas húmedas'].isnull()+df_s['Leyenda_1_5. Superficies de agua'].isnull()]#.plot(x='longitude',y='latitude',column='CH4_column_volume_mixing_ratio_dry_air_bias_corrected')
#sns.scatterplot(data=df_g,x='longitude',y='latitude',s=0.5)#,hue='CH4_column_volume_mixing_ratio_dry_air_bias_corrected')
#df_g.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/glint_intersect_filtered.csv')
#df_s.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/corr_land_2020.csv')
#df_g=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/glint_intersect_filtered.csv')
sns.scatterplot(data=df_s,x='longitude',y='latitude',s=0.5)
df_s

In [ ]:
df_s['coordinate_x']=df_s['longitude'].apply(lambda x: [x])
df_s['coordinate_y']=df_s['latitude'].apply(lambda x: [x])
df_s['coordinates']=df_s['coordinate_x']+df_s['coordinate_y']
delta_y=0.01
delta_x=0.01
df_s['geometry'] = df_s['coordinates'].apply(
    lambda x: Polygon([
        (x[0] - delta_x, x[1] - delta_y),
        (x[0] - delta_x, x[1] + delta_y),
        (x[0] + delta_x, x[1] + delta_y),
        (x[0] + delta_x, x[1] - delta_y)
    ]))# Corregir el delta
df_s

In [ ]:
l_in=list(encoded_df45.columns)
while not l_in[0]=='Leyenda_3_1.1.1. Tejido urbano continuo':
    l_in.pop(0)#(list(dfg.columns))
l_in.remove('groupb')
for i in l_in:
    encoded_df45[i]=encoded_df[i]*encoded_df['SHAPE_Area']
#df_s.loc[:,l_in]=None
#df_s['Leyenda_1_4. Áreas húmedas'][0]

In [ ]:
#df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020_filtered_pausa_349827.csv')


In [ ]:
# for j in df_s.index:
#     #print(j)
#     f=encoded_df45[df_s['geometry'][j].intersects(encoded_df45['geometry'])]
#     f1=f.groupby('groupb')[l_in].aggregate('sum') # Se suma el área total intersectada
#     if not len(f1)==0:
#       for i in l_in:
#         df_s.loc[j, i]=f1[i].iloc[0].copy()
#     if j%5e4==0:
#       print(j)
listind=[]
for j in encoded_df45.index:
    f=df_s[encoded_df45['geometry'][j].intersects(df_s['geometry'])]
    #f1=f.groupby('groupb')[l_in].aggregate('sum') # Se suma el área total intersectada
    #listind.append(f.index)
    df_s.drop(f.index,inplace=True)
    #

#df_sfil=df_s[~listind]
df_s.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020_filtered.csv')
#correr hasta acá

In [ ]:
# listindc=listind[0]
# for i in range(1,len(listind)):
#   #listindc=listindc.append(listind[i])
#   try:
#     df_s.drop(listind[i],inplace=True)
#   except: KeyError
df_s.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020_filtered.csv')

In [ ]:
sns.scatterplot(data=df_s,x='longitude',y='latitude',s=0.5)

In [ ]:
df_s.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020_filtered.csv')


In [ ]:
df_g=df_s[df_s['Leyenda_1_5. Superficies de agua'].isnull()+df_s['Leyenda_1_5. Superficies de agua'].isnull()]

In [ ]:
sns.scatterplot(data=df_g,x='longitude',y='latitude',s=0.5)#,hue='label',s=1,palette=sns.color_palette("tab10"))

In [ ]:
df_g.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_filtered.csv')

In [ ]:
df_g.dtypes.head(60)

In [ ]:
#Correr desde acá 11/11/24
df_g=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_filtered.csv')
df_g['coordinate_x']=df_g['longitude'].apply(lambda x: [x])
df_g['coordinate_y']=df_g['latitude'].apply(lambda x: [x])
df_g['coordinates']=df_g['coordinate_x']+df_g['coordinate_y']
df_g
df_g.coordinates

In [ ]:
df_completo=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_result_corrected.csv')
df_completo

In [ ]:
df_completo['coordinate_x']=df_completo['longitude'].apply(lambda x: [x])
df_completo['coordinate_y']=df_completo['latitude'].apply(lambda x: [x])
df_completo['coordinates']=df_completo['coordinate_x']+df_completo['coordinate_y']
df_completo

In [ ]:
#df_completo.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/colombia_results.csv')
# df_completo=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/colombia_results.csv')
# df_completo

In [ ]:
#df_completo.coordinates.to_numpy()[0]
df_completo_ng=df_completo[df_completo.coordinates.isin(df_g.coordinates)]#Se filtran los datos para saber cuáles son los válidos sin glint
df_completo_ng


In [ ]:
df_completo_ng.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_result_filtered.csv')

In [ ]:
#df_completo_ng.to_csv('/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_result_filtered.csv')
#df_completo.coordinates.isin(df_g.coordinates)

In [ ]:
df_completo_ng.dtypes

In [ ]:
df_completo_ng=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_result_corrected.csv')
df_completo_ng['date']=pd.to_datetime(df_completo_ng['date'])
#df_comleto_ng['year']=df_completo_ng['date'].dt.year
df_2020=df_completo_ng[df_completo_ng.date.dt.year==2020]
df_2020.groupby(['longitude','latitude']).mean('CH4_column_volume_mixing_ratio_dry_air_bias_corrected').to_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020_filtered.csv')

In [ ]:
df_s=pd.read_csv(r'/content/drive/Shareddrives/PIGCC/Satelite/datos_csv/datos corregidos/colombia_prom_2020.csv')
df_s

In [ ]:
#df_s[(df_s['longitude']==-71.495)*(df_s['latitude']==11.925)].index
df_s['geometry'][0].intersects(encoded_df45['geometry'])

In [ ]:
encoded_df45['geometry'][217].intersects(df_s['geometry'])